# 04 · Fine-tune Agent 2 on Colab GPU (PRD §12, gate G2)
**Runs:** Google Colab with a **T4 GPU** — either in the browser or from VS Code with Google's
official *Colab* extension (open this notebook → *Select Kernel* → *Colab* → *New Colab Server* → GPU).

The same training also runs locally on a small GPU via `training/scripts/train_all.py`
(384 tokens, Adafactor). This notebook uses the full profile (512 tokens, AdamW) so the two
models can be compared on **identical product-grouped folds** (`GroupKFold` is deterministic).

**Inputs** (pick one in the next cell):
- `REPO_URL` — your GitHub repo **plus** `labels_v1.jsonl` uploaded next to this notebook, or
- `dist/vera_colab_bundle.zip` from `training/scripts/make_colab_bundle.py`, uploaded to
  `/content` (Files panel) or Drive `MyDrive/vera/`.

**Outputs** (zipped to `/content/vera_colab_outputs.zip` for download, copied to Drive if mounted):
`experience_oof_colab.csv`, `experience_model_colab.onnx`, `experience_tokenizer_colab.json`,
`experience_export_colab.json`. Put them in `data/labels/` and `data/models/` locally, then run
`training/scripts/compare_experience_models.py`.

In [ ]:
# --- Environment ------------------------------------------------------------------
import os, sys, subprocess, zipfile, pathlib

REPO_URL = ""                      # e.g. "https://github.com/you/vera.git" (optional)
USE_DRIVE = False                  # True: read bundle from / write outputs to MyDrive/vera
ROOT = pathlib.Path("/content/vera")
DRIVE = pathlib.Path("/content/drive/MyDrive/vera")

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if not ROOT.exists():
    bundle = next((p for p in [pathlib.Path("/content/vera_colab_bundle.zip"),
                               DRIVE / "vera_colab_bundle.zip"] if p.exists()), None)
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    elif bundle is not None:
        zipfile.ZipFile(bundle).extractall(ROOT)
    else:
        raise FileNotFoundError("Set REPO_URL or upload vera_colab_bundle.zip to /content")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(ROOT / "training/requirements-colab.txt")], check=True)
os.chdir(ROOT)
sys.path[:0] = [str(ROOT / "backend"), str(ROOT / "training")]
os.environ.setdefault("DATABASE_URL", "sqlite:///:memory:")
os.environ.setdefault("REDIS_URL", "redis://localhost:6379/0")

import torch
assert torch.cuda.is_available(), "Switch the runtime to a GPU (T4)"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import json
import pandas as pd
from vera_training.io import validate_reviews

labels_path = next(p for p in [ROOT / "data/labels/labels_v1.jsonl",
                               pathlib.Path("/content/labels_v1.jsonl"),
                               DRIVE / "labels_v1.jsonl"] if p.exists())
rows = [json.loads(line) for line in labels_path.open(encoding="utf-8")]
labelled = validate_reviews(pd.DataFrame(rows), labels=True)
print(len(labelled), "labelled reviews,", labelled.parent_asin.nunique(), "products")
WORK = (DRIVE if USE_DRIVE else pathlib.Path("/content")) / "vera_work"
OUT = pathlib.Path("/content/vera_colab_outputs"); OUT.mkdir(exist_ok=True)

## Cross-fitted out-of-fold predictions (5 product-grouped folds, resumable)

In [ ]:
from vera_training.finetune import FinetuneConfig, crossfit_predictions

config = FinetuneConfig()          # 512 tokens, AdamW, batch 8 (T4 profile)
oof = crossfit_predictions(labelled, WORK / "crossfit", config, n_folds=5)
out = pd.DataFrame({"review_id": labelled.review_id, "experience_raw": oof})
out.to_csv(OUT / "experience_oof_colab.csv", index=False)
print("OOF MAE:", float((out.experience_raw.clip(1, 5) - labelled.experience_score).abs().mean()))

## Final model + ONNX export with parity check (G2: val MAE < 0.50, FP32 ONNX Δ < 0.01)

In [ ]:
import shutil
from vera_training.export import export_experience
from vera_training.finetune import train_final

final_dir = WORK / "final" / "model"
if not (final_dir / "config.json").exists():
    final_dir = train_final(labelled, WORK / "final", config)
validation = json.loads((final_dir / "vera_training.json").read_text())["validation"]
export_dir = pathlib.Path("/content/export"); export_dir.mkdir(exist_ok=True)
report = export_experience(final_dir, export_dir, labelled.sample(32, random_state=0).text.tolist(),
                           "/content/onnx_work", config.max_length)
report["validation"] = validation
shutil.copy(export_dir / "experience_model.onnx", OUT / "experience_model_colab.onnx")
shutil.copy(export_dir / "experience_tokenizer.json", OUT / "experience_tokenizer_colab.json")
(OUT / "experience_export_colab.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))

In [ ]:
archive = shutil.make_archive("/content/vera_colab_outputs", "zip", OUT)
if USE_DRIVE:
    shutil.copy(archive, DRIVE / "vera_colab_outputs.zip")
print("Download:", archive)
try:
    from google.colab import files
    files.download(archive)
except Exception:
    print("Use the Colab Files panel to download it.")